In [ ]:
ヒートマップは以下のコードの内容を変えずに動くようにして
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import os
import json
import zipfile

# =============================================================================
# Unified Thesis Deliverables Builder (Full Code)
#
# Updates (per your request):
#  1) Display names: ppr -> PPR, knn -> kNN (everywhere in outputs/plots/labels)
#  2) Heatmaps: 4 figures only (CV/OOD × R2/RMSE)
#     - matrix is 10 models × (A–F × 4 targets) = 10 × 24
#     - x-axis: bottom row shows targets (PCE, Jsc, Voc, FF) repeated
#     - dataset labels A–F appear as a second (upper) row centered over each group
#     - coloring: matplotlib default (no custom colormap specified)
# =============================================================================

# --- 1. Settings --------------------------------------------------------------
base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression"

output_main_dir = "Final_Thesis_Deliverables_Full_Pack"
os.makedirs(output_main_dir, exist_ok=True)

# Input dataset keys (used internally)
target_datasets = ["base", "base_OH", "base_FP", "all", "all_OH", "all_FP"]

# Targets in required order for heatmap x-axis
target_vars = ["PCEmax", "Jsc", "Voc", "FF"]

regions = ["Inner", "Outer"]  # Inner=CV10, Outer=OOD
panel_labels = ["A", "B", "C", "D", "E", "F"]

THRES_UP = 1.05
EPSILON = 1e-6

# Type1 integer parameters
INT_PARAMS = {"ncomp", "C", "degree", "nterms", "k", "min_samples_leaf", "hidden_layer_sizes"}

model_configs = {
    "PLS":           {"dir": "PLS",           "sub": "20251218_Rebuilt_12Files_PLS_Train_CV10OOF_OOD",            "file": "fixed_20251218_PLS_summary.csv",                  "p": ["ncomp"]},
    "svmLinear":     {"dir": "svmLinear",     "sub": "20251218_Rebuilt_12Files_SVM_Linear_Train_CV10OOF_OOD",    "file": "fixed_20251218_SVM_Linear_summary.csv",           "p": ["C"]},
    "gaussprLinear": {"dir": "gaussprLinear", "sub": "20251218_Rebuilt_12Files_GPR_Linear_Train_CV10OOF_OOD",    "file": "fixed_20251218_GPR_Linear_summary.csv",           "p": []},
    "svmRadial":     {"dir": "SVMRadial",     "sub": "20251218_Rebuilt_12Files_SVM_Radial_Train_CV10OOF_OOD",    "file": "fixed_20251218_SVM_Radial_summary.csv",           "p": ["sigma", "C"]},
    "gaussprRadial": {"dir": "gaussPrRadial", "sub": "20251218_Rebuilt_12Files_GPR_Radial_Train_CV10OOF_OOD",    "file": "fixed_20251218_GPR_Radial_summary.csv",           "p": ["sigma"]},
    "gcvEarth":      {"dir": "gcvEarth",      "sub": "20251224_Rebuilt_12Files_gcvEarth_Strict_Final",           "file": "fixed_20251224_gcvEarth_summary.csv",             "p": ["degree"]},
    "ppr":           {"dir": "ppr",           "sub": "20251218_Rebuilt_12Files_PPR_Train_CV10OOF_OOD",            "file": "fixed_20251218_PPR_summary.csv",                  "p": ["nterms"]},
    "knn":           {"dir": "knn",           "sub": "20251218_Rebuilt_12Files_kNN_Train_CV10OOF_OOD",            "file": "fixed_20251218_kNN_summary.csv",                  "p": ["k"]},
    "Rborist":       {"dir": "Rborist",       "sub": "20251218_Rebuilt_12Files_RboristLikeRF_Train_CV10OOF_OOD", "file": "fixed_20251218_RboristLikeRF_summary.csv",         "p": ["max_features", "min_samples_leaf"]},
    "monmlp":        {"dir": "monmlp",        "sub": "20251218_Rebuilt_12Files_monmlp_Train_CV10OOF_OOD",         "file": "fixed_20251218_monmlp_summary.csv",               "p": ["alpha", "hidden_layer_sizes"]},
}

ordered_models = list(model_configs.keys())

# Display-name mapping (requested)
MODEL_DISPLAY = {
    "ppr": "PPR",
    "knn": "kNN",
}
def disp_model_name(m: str) -> str:
    return MODEL_DISPLAY.get(m, m)

ordered_models_disp = [disp_model_name(m) for m in ordered_models]

# Heatmap dataset labels: A–F
dataset_AF = ["A", "B", "C", "D", "E", "F"]

# Heatmap target labels (requested: PCE, Jsc, Voc, FF order)
# You requested "PCE", so we'll display "PCE" (coming from PCEmax).
target_short = ["PCE", "Jsc", "Voc", "FF"]

# --- 2. Utilities -------------------------------------------------------------
def format_val(v):
    """Format numbers: 3 decimals, scientific in LaTeX for very small."""
    try:
        val = float(v)
        if val == 0:
            return "0.000"
        abs_v = abs(val)
        if abs_v >= 0.001:
            return f"{val:.3f}"
        formatted = "{:.2e}".format(val)
        base, exp = formatted.split("e")
        return f"${float(base):.2f} \\times 10^{{{int(exp)}}}$"
    except:
        return str(v)

def get_dataset_key(filename):
    fn = str(filename).lower()
    if "base" in fn:
        if "oh" in fn: return "base_OH"
        if "fp" in fn: return "base_FP"
        return "base"
    if "all" in fn:
        if "oh" in fn: return "all_OH"
        if "fp" in fn: return "all_FP"
        return "all"
    return "unknown"

def parse_param(p_str, p_name):
    """Extract param from Best_Params; integer-ize selected params."""
    if pd.isna(p_str) or p_str == "":
        return "N/A"
    p_str = str(p_str).strip()
    extracted = "N/A"

    if p_str.startswith("{"):
        try:
            d = json.loads(p_str.replace("'", '"'))
            if p_name in d:
                extracted = str(d[p_name])
        except:
            pass

    if extracted == "N/A":
        pattern = rf"{p_name}[^0-9\.\-]*[=:]?[^0-9\.\-]*(\([^\)]+\)|[0-9\.eE\-\+]+)"
        m = re.search(pattern, p_str)
        if m:
            extracted = m.group(1).rstrip(",")

    if extracted != "N/A":
        s = str(extracted).strip()

        # keep tuple-like as-is
        if s.startswith("(") and s.endswith(")"):
            return s

        try:
            fv = float(s)
            if p_name in INT_PARAMS:
                return str(int(round(fv)))
            return format_val(fv)
        except:
            if p_name in INT_PARAMS:
                mm = re.fullmatch(r"\s*([0-9]+)\s*", s)
                if mm:
                    return mm.group(1)
            return s

    return "N/A"

def annotate_nonoverlap(ax, xs, ys, labels, *,
                        base_px=18, step_px=10, max_rings=18, angles=16,
                        fontsize=11, fontweight="bold"):
    """Place labels by searching empty directions; arrow length expands."""
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    placed_bboxes = []
    angle_list = np.linspace(0, 2*np.pi, angles, endpoint=False)

    for x, y, txt in zip(xs, ys, labels):
        chosen = None
        for ring in range(max_rings):
            r = base_px + ring * step_px
            for ang in angle_list:
                dx = r * np.cos(ang)
                dy = r * np.sin(ang)

                ann = ax.annotate(
                    txt,
                    xy=(x, y),
                    xytext=(dx, dy),
                    textcoords="offset points",
                    fontsize=fontsize,
                    fontweight=fontweight,
                    ha="left" if dx >= 0 else "right",
                    va="bottom" if dy >= 0 else "top",
                    arrowprops=dict(arrowstyle="-", color="gray", lw=1),
                )
                fig.canvas.draw()
                bb = ann.get_window_extent(renderer=renderer).expanded(1.02, 1.12)

                collide = any(bb.overlaps(prev) for prev in placed_bboxes)
                ax_bb = ax.get_window_extent(renderer=renderer)
                too_out = (not ax_bb.contains(bb.x0, bb.y0)) or (not ax_bb.contains(bb.x1, bb.y1))

                if (not collide) and (not too_out):
                    chosen = (ann, bb)
                    break
                ann.remove()

            if chosen is not None:
                break

        if chosen is None:
            ann = ax.annotate(
                txt, xy=(x, y),
                xytext=(base_px + max_rings*step_px, base_px + max_rings*step_px),
                textcoords="offset points",
                fontsize=fontsize, fontweight=fontweight,
                arrowprops=dict(arrowstyle="-", color="gray", lw=1),
            )
            fig.canvas.draw()
            bb = ann.get_window_extent(renderer=renderer).expanded(1.02, 1.12)
            chosen = (ann, bb)

        placed_bboxes.append(chosen[1])

# --- 3. Step1: Load -> master_df ---------------------------------------------
print("Step 1: Loading raw data...")
all_rows = []
loaded_files = []
missing_files = []

for mod, cfg in model_configs.items():
    path = os.path.join(base_path, cfg["dir"], cfg["sub"], cfg["file"])
    if os.path.exists(path):
        loaded_files.append((mod, path))
        df = pd.read_csv(path)

        required_cols = ["Target", "File", "CV10_R2", "CV10_RMSE", "OOD_R2", "OOD_RMSE", "Best_Params"]
        missing_cols = [c for c in required_cols if c not in df.columns]
        if missing_cols:
            raise ValueError(f"[ERROR] Missing columns in {path}: {missing_cols}")

        for _, row in df.iterrows():
            file_str = str(row["File"])
            imp = file_str[0] if len(file_str) > 0 else "?"
            all_rows.append({
                "Model": mod,  # internal
                "Target": row["Target"],
                "Dataset": get_dataset_key(row["File"]),
                "Imp": imp,  # expects 'n' / 'm'
                "Inner_R2": row["CV10_R2"],
                "Inner_RMSE": row["CV10_RMSE"],
                "Outer_R2": row["OOD_R2"],
                "Outer_RMSE": row["OOD_RMSE"],
                "Best_Params": row["Best_Params"],
            })
    else:
        missing_files.append((mod, path))

master_df = pd.DataFrame(all_rows)

print("\n[INFO] Loaded model files:")
for mod, p in loaded_files:
    print(f"  - {mod} ({disp_model_name(mod)}): {p}")

if missing_files:
    print("\n[WARN] Missing model files:")
    for mod, p in missing_files:
        print(f"  - {mod} ({disp_model_name(mod)}): {p}")

print(f"\n[INFO] master_df rows = {len(master_df)}")
if len(master_df) == 0:
    raise RuntimeError("[ERROR] master_df is empty. No input summary CSVs were loaded.")

# --- 4. Step2: Type1 Summary (16 files) --------------------------------------
print("Step 2: Generating Type 1 Summary files...")
type1_dir = os.path.join(output_main_dir, "Summary_Files_16")
os.makedirs(type1_dir, exist_ok=True)

for reg in regions:
    r_label = "Inner" if reg == "Inner" else "Outer"
    for imp in ["n", "m"]:
        for tgt in target_vars:
            rows = []
            for mod in ordered_models:
                sub = master_df[(master_df["Model"] == mod) & (master_df["Target"] == tgt) & (master_df["Imp"] == imp)]
                if sub.empty:
                    continue

                # score row
                score_row = {"Model": disp_model_name(mod)}
                for ds in target_datasets:
                    m = sub[sub["Dataset"] == ds]
                    if not m.empty:
                        score_row[ds] = f"{format_val(m.iloc[0][f'{r_label}_R2'])} ({format_val(m.iloc[0][f'{r_label}_RMSE'])})"
                    else:
                        score_row[ds] = "N/A"
                rows.append(score_row)

                # parameter rows
                for pn in model_configs[mod]["p"]:
                    p_row = {"Model": f"({pn})"}
                    for ds in target_datasets:
                        m = sub[sub["Dataset"] == ds]
                        p_row[ds] = parse_param(m.iloc[0]["Best_Params"], pn) if not m.empty else "N/A"
                    rows.append(p_row)

            out_path = os.path.join(type1_dir, f"Summary_{reg}_{imp}_{tgt}.csv")
            pd.DataFrame(rows).to_csv(out_path, index=False)

# --- 5. Step3: Type2 Detailed Ratio Matrices ---------------------------------
print("Step 3: Generating Type 2 Detailed Matrices (with 2 judges)...")
type2_dir = os.path.join(output_main_dir, "Detailed_Matrices")
os.makedirs(type2_dir, exist_ok=True)

def build_type2_matrix(metric_kind):
    """
    metric_kind: "R2" or "RMSE"
    Output: 3 rows per model (value row + 5% judge row + binary judge row)
    """
    final_rows = []
    for mod in ordered_models:
        mod_disp = disp_model_name(mod)

        row_v  = {"Model": mod_disp}
        row_j5 = {"Model": f"({mod_disp}_Judge_5pc)"}
        row_jb = {"Model": f"({mod_disp}_Judge_Binary)"}

        for reg in regions:
            prefix = "CV" if reg == "Inner" else "OOD"
            col_metric = f"{reg}_{metric_kind}"  # Inner_R2 / Outer_R2 etc.

            for tgt in target_vars:
                for ds in target_datasets:
                    col = f"{metric_kind}_Ratio_{prefix}_{tgt}_{ds}"

                    sn = master_df[(master_df["Model"]==mod) & (master_df["Target"]==tgt) & (master_df["Dataset"]==ds) & (master_df["Imp"]=="n")]
                    sm = master_df[(master_df["Model"]==mod) & (master_df["Target"]==tgt) & (master_df["Dataset"]==ds) & (master_df["Imp"]=="m")]

                    val, j5, jb = "N/A", 0, -1
                    if not sn.empty and not sm.empty:
                        nv = sn.iloc[0][col_metric]
                        mv = sm.iloc[0][col_metric]

                        if pd.isna(nv) or pd.isna(mv):
                            val, j5, jb = "N/A", 0, -1
                        elif abs(float(nv)) > EPSILON:
                            ratio = float(mv) / float(nv)
                            val = format_val(ratio)

                            if metric_kind == "R2":
                                j5 = 1 if ratio >= THRES_UP else (-1 if ratio <= 1/THRES_UP else 0)
                                jb = 1 if ratio >= 1.0 else -1
                            else:  # RMSE smaller is better
                                j5 = 1 if ratio <= 0.95 else (-1 if ratio >= 1.05 else 0)
                                jb = 1 if ratio <= 1.0 else -1

                    row_v[col], row_j5[col], row_jb[col] = val, j5, jb

        final_rows.extend([row_v, row_j5, row_jb])
    return pd.DataFrame(final_rows)

for m_kind in ["R2", "RMSE"]:
    df2 = build_type2_matrix(m_kind)
    out_path = os.path.join(type2_dir, f"Detailed_Ratio_Matrix_{m_kind}_Combined_Judges.csv")
    df2.to_csv(out_path, index=False)

# --- 5.5 Step3.5: Heatmaps (4 graphs: CV/OOD × R2/RMSE) ----------------------
print("Step 3.5: Generating Heatmaps (4 graphs only: CV/OOD × R2/RMSE)...")
heat_dir = os.path.join(output_main_dir, "Heatmaps_Ratio")
os.makedirs(heat_dir, exist_ok=True)

def ratio_value(mod, tgt, ds, reg, metric_kind):
    sn = master_df[(master_df["Model"]==mod) & (master_df["Target"]==tgt) & (master_df["Dataset"]==ds) & (master_df["Imp"]=="n")]
    sm = master_df[(master_df["Model"]==mod) & (master_df["Target"]==tgt) & (master_df["Dataset"]==ds) & (master_df["Imp"]=="m")]
    if sn.empty or sm.empty:
        return np.nan
    nv = sn.iloc[0][f"{reg}_{metric_kind}"]
    mv = sm.iloc[0][f"{reg}_{metric_kind}"]
    if pd.isna(nv) or pd.isna(mv) or abs(float(nv)) <= EPSILON:
        return np.nan
    return float(mv) / float(nv)

def plot_ratio_heatmap_4panel(reg, metric_kind):
    """
    reg: "Inner"(CV) or "Outer"(OOD)
    metric_kind: "R2" or "RMSE"
    Heatmap matrix:
      - rows: models (10)
      - cols: A–F × targets(4) = 24
      - bottom x labels: PCE/Jsc/Voc/FF repeated
      - upper row: A–F centered over each group of 4 columns
    """
    n_models = len(ordered_models)
    n_cols = len(target_datasets) * len(target_vars)  # 6 * 4 = 24
    mat = np.full((n_models, n_cols), np.nan, dtype=float)

    # Fill matrix: columns ordered by dataset (A..F), within each dataset by target_vars order
    col_labels_bottom = []
    col = 0
    for ds in target_datasets:
        for t_short, tgt in zip(target_short, target_vars):
            col_labels_bottom.append(t_short)
            for i, mod in enumerate(ordered_models):
                mat[i, col] = ratio_value(mod, tgt, ds, reg, metric_kind)
            col += 1

    fig, ax = plt.subplots(figsize=(16, 6))
    im = ax.imshow(mat, aspect="auto")  # default coloring (no explicit cmap)

    # Y labels (horizontal)
    ax.set_yticks(np.arange(n_models))
    ax.set_yticklabels([disp_model_name(m) for m in ordered_models], rotation=0, fontsize=10)

    # X labels bottom: targets repeated
    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(col_labels_bottom, fontsize=9, fontweight="bold")
    ax.tick_params(axis="x", pad=8)

    # Add dataset labels A–F as "upper row" centered above each dataset block of 4 columns
    # Place text at group centers in axis coordinates
    # centers at 1.5, 5.5, 9.5, 13.5, 17.5, 21.5 (0-indexed columns)
    y_text = -0.18  # above ticks (negative = above axis in axes fraction when using transform)
    for gi, A in enumerate(dataset_AF):
        center = gi * len(target_vars) + (len(target_vars)-1) / 2.0
        ax.text(center, 1.02, A, transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=12, fontweight="bold")

    # Vertical separators between datasets
    for gi in range(1, len(target_datasets)):
        ax.axvline(gi * len(target_vars) - 0.5, color="black", lw=0.6, alpha=0.6)

    region_label = "CV" if reg == "Inner" else "OOD"
    ax.set_title(f"Ratio (m/n) Heatmap | {region_label} | {metric_kind}", fontsize=14, fontweight="bold")

    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("m / n", rotation=90, fontsize=12, fontweight="bold")

    plt.tight_layout()
    out_path = os.path.join(heat_dir, f"Heatmap_{region_label}_{metric_kind}.png")
    plt.savefig(out_path, dpi=300)
    plt.close()

# 4 graphs
plot_ratio_heatmap_4panel("Inner", "R2")   # CV_R2
plot_ratio_heatmap_4panel("Inner", "RMSE") # CV_RMSE
plot_ratio_heatmap_4panel("Outer", "R2")   # OOD_R2
plot_ratio_heatmap_4panel("Outer", "RMSE") # OOD_RMSE

# --- 6. Step4: Type3 Statistics ----------------------------------------------
print("Step 4: Generating Type 3 Statistics...")
type3_dir = os.path.join(output_main_dir, "Statistics_Summaries")
os.makedirs(type3_dir, exist_ok=True)

compare_list = []
for (mod, tgt, ds), group in master_df.groupby(["Model", "Target", "Dataset"]):
    rn = group[group["Imp"] == "n"]
    rm = group[group["Imp"] == "m"]
    if not rn.empty and not rm.empty:
        res = {"Model": disp_model_name(mod), "Target": tgt, "Dataset": ds}
        for reg in regions:
            for mtr in ["R2", "RMSE"]:
                nv = rn.iloc[0][f"{reg}_{mtr}"]
                mv = rm.iloc[0][f"{reg}_{mtr}"]
                res[f"n_{mtr}_{reg}"] = nv
                res[f"m_{mtr}_{reg}"] = mv
                if mtr == "R2":
                    res[f"Win_{mtr}_{reg}"] = 1 if float(mv) >= float(nv) - EPSILON else 0
                else:
                    res[f"Win_{mtr}_{reg}"] = 1 if float(mv) <= float(nv) + EPSILON else 0
        compare_list.append(res)

comp_df = pd.DataFrame(compare_list)

for axis in ["Target", "Model", "Dataset"]:
    summary = comp_df.groupby(axis).agg({
        **{f"{i}_{mtr}_{reg}": "mean" for i in ["n", "m"] for mtr in ["R2", "RMSE"] for reg in regions},
        **{f"Win_{mtr}_{reg}": "sum" for mtr in ["R2", "RMSE"] for reg in regions},
        "Model": "count"
    }).rename(columns={"Model": "Total_Tests"})

    for c in summary.columns:
        if c != "Total_Tests":
            summary[c] = summary[c].apply(format_val)

    if axis == "Model":
        summary = summary.reindex(ordered_models_disp)

    out_path = os.path.join(type3_dir, f"Summary_Statistics_By_{axis}.csv")
    summary.to_csv(out_path)

# --- 7. Step5: Panel Scatter Plots -------------------------------------------
print("Step 5: Generating 2x3 Panel Plots (non-overlap labels, dataset-filtered)...")
type4_dir = os.path.join(output_main_dir, "Scatter_Plots_Panels")
os.makedirs(type4_dir, exist_ok=True)

for tgt in target_vars:
    for split in regions:
        for m_kind in ["R2", "RMSE"]:
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))
            axes = axes.flatten()

            for i, ds in enumerate(target_datasets):
                ax = axes[i]

                sub_n = master_df[(master_df["Target"] == tgt) & (master_df["Imp"] == "n") & (master_df["Dataset"] == ds)]
                sub_m = master_df[(master_df["Target"] == tgt) & (master_df["Imp"] == "m") & (master_df["Dataset"] == ds)]

                x_v, y_v, names = [], [], []
                for mod in ordered_models:
                    rn = sub_n[sub_n["Model"] == mod]
                    rm = sub_m[sub_m["Model"] == mod]
                    if not rn.empty and not rm.empty:
                        vn = rn.iloc[0][f"{split}_{m_kind}"]
                        vm = rm.iloc[0][f"{split}_{m_kind}"]
                        if not (pd.isna(vn) or pd.isna(vm)):
                            x_v.append(float(vn))
                            y_v.append(float(vm))
                            names.append(disp_model_name(mod))

                ax.text(0.05, 0.95, panel_labels[i], transform=ax.transAxes,
                        fontsize=28, fontweight="bold", va="top")
                ax.set_title(ds, fontsize=16, fontweight="bold")
                ax.set_xlabel(f"{m_kind} (n)", fontsize=14, fontweight="bold")
                ax.set_ylabel(f"{m_kind} (m)", fontsize=14, fontweight="bold")

                if not x_v:
                    ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
                    continue

                ax.scatter(x_v, y_v, s=100, edgecolors="black", alpha=0.85)

                if m_kind == "R2":
                    lims = [0.0, 1.0]
                else:
                    allv = x_v + y_v
                    mn = min(allv)
                    mx = max(allv)
                    if mx == mn:
                        lims = [mn - 0.5, mx + 0.5]
                    else:
                        pad = 0.1 * (mx - mn)
                        lims = [mn - pad, mx + pad]

                ax.plot(lims, lims, linestyle="--", alpha=0.6)
                ax.set_xlim(lims)
                ax.set_ylim(lims)
                ax.set_aspect("equal", adjustable="box")

                annotate_nonoverlap(ax, x_v, y_v, names, fontsize=11, fontweight="bold")

            plt.tight_layout()
            out_path = os.path.join(type4_dir, f"Panel_{tgt}_{split}_{m_kind}.png")
            plt.savefig(out_path, dpi=300)
            plt.close()

# --- 8. Step6: ZIP ------------------------------------------------------------
print("Step 6: Zipping all results...")
zip_path = os.path.join(base_path, "Final_Thesis_Deliverables_Unified.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(output_main_dir):
        for file in files:
            full = os.path.join(root, file)
            rel = os.path.relpath(full, output_main_dir)
            zipf.write(full, rel)

print(f"\n=== ALL SUCCESS ===\nResults saved to folder: {output_main_dir}\nZip created at: {zip_path}")

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def create_styled_heatmap(file_path, title, output_img, is_rmse=False):
    df = pd.read_csv(file_path)
    
    # Extract Model Names (those not starting with '(')
    model_rows = df[~df['Model'].str.startswith('(')].copy()
    
    # Extract Judge 5pc rows
    judge_rows = df[df['Model'].str.contains('Judge_5pc')].copy()
    
    # Set Model name as index for both
    model_rows.set_index('Model', inplace=True)
    judge_rows.set_index('Model', inplace=True)
    
    # Ensure they align (extract the core model name from judge index)
    judge_rows.index = judge_rows.index.str.extract(r'\((.*)_Judge_5pc\)')[0]
    
    # Convert to numeric, handle 'N/A'
    val_df = model_rows.apply(pd.to_numeric, errors='coerce')
    judge_df = judge_rows.apply(pd.to_numeric, errors='coerce')
    
    # Reorder models if possible to match a standard order
    pref_order = ["PLS", "svmLinear", "gaussprLinear", "svmRadial", "gaussprRadial", "gcvEarth", "ppr", "knn", "Rborist", "monmlp"]
    available_models = [m for m in pref_order if m in val_df.index]
    val_df = val_df.loc[available_models]
    judge_df = judge_df.loc[available_models]

    # Plotting
    plt.figure(figsize=(20, 8))
    
    # We use a custom color map for the background based on Judge values
    # But seaborn heatmap takes one array for color. 
    # We can use Judge values for colors and annotate with the Ratio values.
    
    # -1: Red (Loss), 0: White (Neutral), 1: Green (Win)
    from matplotlib.colors import LinearSegmentedColormap
    colors = ["#ff9999", "#ffffff", "#99ff99"] # Red, White, Green
    cmap = LinearSegmentedColormap.from_list("judge_cmap", colors, N=3)

    sns.heatmap(judge_df, annot=val_df, fmt=".3f", cmap=cmap, center=0, 
                linewidths=.5, cbar_kws={'label': 'Judgment (-1: Loss, 0: Neutral, 1: Win)'},
                annot_kws={"size": 10})
    
    plt.title(title, fontsize=16, pad=20)
    plt.xlabel("Target Variable & Descriptor Dataset", fontsize=12)
    plt.ylabel("ML Model", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig(output_img, dpi=300)
    plt.close()

# Files
f_r2 = "Detailed_Ratio_Matrix_R2_Unified_onlyCV.csv"
f_rmse = "Detailed_Ratio_Matrix_RMSE_Unified _onlyCV.csv"

create_styled_heatmap(f_r2, "MICE Imputation Effect: R2 Ratio (m_R2 / n_R2) - CV Results", "Heatmap_R2_Ratio.png")
create_styled_heatmap(f_rmse, "MICE Imputation Effect: RMSE Ratio (m_RMSE / n_RMSE) - CV Results", "Heatmap_RMSE_Ratio.png", is_rmse=True)

print("Heatmaps created.")